## ![](https://ga-dash.s3.amazonaws.com/production/assets/logo-9f88ae6c9c3871690e33280fcf557f33.png) Project 3: NLP Classification: Subreddit Pepsi vs Coca-Cola | Part 4: Model Tuning

---

[README](../README.md) | [Part 1: EDA](01_EDA.ipynb) | [Part 2: Vectorizer](02_Vectorizer.ipynb) | [Part 3: Vectorizer Performance](03_Vectorizer_Performance.ipynb) | **Part 4: Model Tuning**

---

### Introduction
From [Part 3: Vectorizer Performance](03_Vectorizer_Performance.ipynb), we learned about the impact of vectorizers, tokenizers, and their parameters on the model's accuracy. However, since we only used the default model settings, there is potential to improve performance by tuning the model's parameters. In this chapter, we will explore how to optimize the model by adjusting these parameters.

### Import

In [251]:
# Data manipulation and plotting
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# NLP tools
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize, RegexpTokenizer
from nltk.stem import PorterStemmer, WordNetLemmatizer
import nltk
# nltk.download('punkt')

# Model selection and evaluation
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.metrics import make_scorer, accuracy_score, confusion_matrix, recall_score, precision_score, f1_score
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer

# Classifiers
from sklearn.naive_bayes import MultinomialNB 
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import BaggingClassifier, RandomForestClassifier, AdaBoostClassifier, GradientBoostingClassifier
from sklearn.svm import SVC
from xgboost import XGBClassifier

# Time utility
import time
from joblib import parallel_backend

# Set seed for reproducibility
np.random.seed(42)             

### Model Selection
- The goal of the model is to predict outcomes effectively, so we select the model with the highest testing F1 score.
- The chosen model is Gradient Boosting having **74.5% on F1 score**, which achieved an accuracy of **81.1% on the training set** and **69.9% on the testing set**.
- Its parameters are as follows:
    - Vectorizer: `CountVectorizer`
    - Tokenization: Performed using **lowering**, with English **stopwords removed**
    - Features: Unigrams and bigrams, with a maximum of `3000` features, `min_df = 3`, and `max_df = 0.9`

In [252]:
df1 = pd.read_csv('../data/cv_results_nb.csv')                       # Load Data Naive Bayes Performance
df2 = pd.read_csv('../data/cv_results_xgb.csv')                      # Load Data XGBoost Performance 
df3 = pd.read_csv('../data/cv_results_ada.csv')                      # Load Data AdaBoost Performance 
df4 = pd.read_csv('../data/cv_results_gb.csv')                       # Load Data GradientBoost Performance 
# Check shape
df1.shape, df2.shape, df3.shape, df4.shape                                                      

((288, 73), (288, 73), (288, 73), (288, 73))

In [253]:
df = pd.concat([df1, df2, df3, df4], axis = 0, ignore_index = True)

columns = ['param_classifier' 
           , 'param_vectorizer'
           , 'param_vectorizer__tokenizer' 
           , 'param_vectorizer__max_features' 
           , 'param_vectorizer__ngram_range' 
           , 'param_vectorizer__min_df' 
           , 'param_vectorizer__max_df'
           , 'mean_train_accuracy' 
           , 'mean_test_accuracy'
           , 'mean_train_f1' 
           , 'mean_test_f1'
          ]

df = df[columns]

df.sort_values(by = 'mean_test_f1', ascending = False).head(1)

,param_classifier,param_vectorizer,param_vectorizer__tokenizer,param_vectorizer__max_features,param_vectorizer__ngram_range,param_vectorizer__min_df,param_vectorizer__max_df,mean_train_accuracy,mean_test_accuracy,mean_train_f1,mean_test_f1
955,GradientBoostingClassifier(),CountVectorizer(token_pattern=None),<function lower_stop at 0x00000236FE7A89A0>,3000.0,"(1, 2)",3,0.9,0.810983,0.699263,0.834754,0.745306


### Model Tuning

In [254]:
# Set Features and Target

df = pd.read_csv('../data/subreddit_pepsi_vs_cocacola_clean.csv')          # Load Data
df.head(1)                                                                 # Check first row

X = df['title_body']
y = df['is_pepsi']

# Split the data to training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X
                                                    , y
                                                    , test_size = 0.2
                                                    , stratify = y
                                                   )

In [255]:
# Related words are words that can easily identify the subreddit categories, 
# such as the names of brands.
related_words = {'pepsi', 'pepsico', 'coca', 'cola', 'coke'}

# Custom stop words are common English words that are meaningless, 
# and we also include related words to create a custom stop word list.

custom_stop_words = set(stopwords.words('english')).union(related_words)

In [256]:
# Set Pipeline

def lower_stop(doc):
    doc = doc.lower()
    tokens = word_tokenize(doc)
    return [token for token in tokens if token not in custom_stop_words and token.isalpha()]

params = {
    # ----------------------------------------------------- Parameters of Vectorizer having maximum test's f1 score
    'vectorizer': [CountVectorizer(token_pattern = None)] 
    , 'vectorizer__tokenizer': [lower_stop] 
    , 'vectorizer__max_features': [3000]
    , 'vectorizer__ngram_range': [(1, 2)]
    , 'vectorizer__min_df': [3]
    , 'vectorizer__max_df': [0.9]
    # -----------------------------------------------------
    , 'classifier': [GradientBoostingClassifier()]
    , 'classifier__learning_rate': [0.1,0.2,0.3]
    , 'classifier__n_estimators': [100,200]
    , 'classifier__subsample': [0.5,1.0]
    , 'classifier__max_depth': [2,3,4]
}


# Default Parameter of GradientBoostingClassifier
# GradientBoostingClassifier(
#     *,
#     loss='log_loss',
#     learning_rate=0.1,
#     n_estimators=100,
#     subsample=1.0,
#     criterion='friedman_mse',
#     min_samples_split=2,
#     min_samples_leaf=1,
#     min_weight_fraction_leaf=0.0,
#     max_depth=3,
#     min_impurity_decrease=0.0,
#     init=None,
#     random_state=None,
#     max_features=None,
#     verbose=0,
#     max_leaf_nodes=None,
#     warm_start=False,
#     validation_fraction=0.1,
#     n_iter_no_change=None,
#     tol=0.0001,
#     ccp_alpha=0.0,
# )

In [257]:
# Custom metrics used in GridSearch results
scorers = {
    'accuracy': make_scorer(accuracy_score)
    , 'recall': make_scorer(recall_score, average = 'binary')
    , 'precision': make_scorer(precision_score, average = 'binary')
    , 'f1': make_scorer(f1_score, average = 'binary')
}

In [258]:
# Create Pipeline
pipeline = Pipeline([
    ('vectorizer', CountVectorizer())                    # Will replace in GridSearch  
    , ('classifier', MultinomialNB())                    # Will replace in GridSearch 
])

In [259]:
# GridSearchCV
grid_search = GridSearchCV(pipeline
                           , param_grid=params
                           , cv = 5
                           , verbose = 2
                           , scoring = scorers
                           , refit = 'f1'
                           , return_train_score = True
)

In [260]:
# Fit the grid search

# Record start time
start_time = time.time()

with parallel_backend('threading', n_jobs=-1):
    grid_search.fit(X_train, y_train)

Fitting 5 folds for each of 36 candidates, totalling 180 fits
[CV] END classifier=GradientBoostingClassifier(), classifier__learning_rate=0.1, classifier__max_depth=2, classifier__n_estimators=100, classifier__subsample=1.0, vectorizer=CountVectorizer(token_pattern=None), vectorizer__max_df=0.9, vectorizer__max_features=3000, vectorizer__min_df=3, vectorizer__ngram_range=(1, 2), vectorizer__tokenizer=<function lower_stop at 0x000002A9E6F136A0>; total time=   4.8s
[CV] END classifier=GradientBoostingClassifier(), classifier__learning_rate=0.1, classifier__max_depth=2, classifier__n_estimators=100, classifier__subsample=1.0, vectorizer=CountVectorizer(token_pattern=None), vectorizer__max_df=0.9, vectorizer__max_features=3000, vectorizer__min_df=3, vectorizer__ngram_range=(1, 2), vectorizer__tokenizer=<function lower_stop at 0x000002A9E6F136A0>; total time=   5.2s
[CV] END classifier=GradientBoostingClassifier(), classifier__learning_rate=0.1, classifier__max_depth=2, classifier__n_estima

In [261]:
# Record end time
end_time = time.time()

# Calculate elapsed time
elapsed_time = end_time - start_time
print(f"Time taken: {elapsed_time:.4f} seconds")

Time taken: 161.4663 seconds


In [262]:
# Collect results
cv_results = pd.DataFrame(grid_search.cv_results_)
cv_results.to_csv('../data/cv_results_gb_tuning.csv') 

In [268]:
# Display important metrics
cv_results.sort_values(by = 'mean_test_f1', ascending = False).head(1)

,mean_fit_time,std_fit_time,mean_score_time,std_score_time,param_classifier,param_classifier__learning_rate,param_classifier__max_depth,param_classifier__n_estimators,param_classifier__subsample,param_vectorizer,...,mean_test_f1,std_test_f1,rank_test_f1,split0_train_f1,split1_train_f1,split2_train_f1,split3_train_f1,split4_train_f1,mean_train_f1,std_train_f1
5,6.88425,0.325249,0.575723,0.31931,GradientBoostingClassifier(),0.1,3,100,1.0,CountVectorizer(token_pattern=None),...,0.73881,0.028768,1,0.835779,0.834136,0.842547,0.823115,0.83275,0.833666,0.006257


In [264]:
# Display the best parameters (giving highest f1-score on testing set) and corresponding metrics

print(f"The Best parameters: {grid_search.best_params_}")
print(f"Testing F1 score: {grid_search.cv_results_['mean_test_f1'][grid_search.best_index_]:.3f}")
print(f"Training F1 score: {grid_search.cv_results_['mean_train_f1'][grid_search.best_index_]:.3f}")
print(f"Testing accuracy: {grid_search.cv_results_['mean_test_accuracy'][grid_search.best_index_]:.3f}")
print(f"Training accuracy: {grid_search.cv_results_['mean_train_accuracy'][grid_search.best_index_]:.3f}")

The Best parameters: {'classifier': GradientBoostingClassifier(), 'classifier__learning_rate': 0.1, 'classifier__max_depth': 3, 'classifier__n_estimators': 100, 'classifier__subsample': 1.0, 'vectorizer': CountVectorizer(token_pattern=None), 'vectorizer__max_df': 0.9, 'vectorizer__max_features': 3000, 'vectorizer__min_df': 3, 'vectorizer__ngram_range': (1, 2), 'vectorizer__tokenizer': <function lower_stop at 0x000002A9E6F136A0>}
Testing F1 score: 0.739
Training F1 score: 0.834
Testing accuracy: 0.692
Training accuracy: 0.810


In [266]:
# Rebuild the pipeline using the best parameters
pipeline.set_params(**best_params)

# Fit the model with the best parameters on the entire training dataset
pipeline.fit(X_train, y_train)

# Evaluate the model on the test set
test_accuracy = pipeline.score(X_test, y_test)
print(f"Final model testing accuracy: {test_accuracy:.3f}")

Final model testing accuracy: 0.679
